In [1]:
# ============================================================
# 0. Import libraries
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# 如果你本地支持中文字体，可以打开这个
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. Read Excel
# ============================================================
project_dir = Path.cwd()
file_path = project_dir / "4_month_data_2026_02_01_2026_06_25.xlsx"
#file_path = r"C:\Users\fshhan17\Desktop\9_2025_12_15_2025_06_25.xlsx"
sheet_name = "Sheet1"

raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, engine="openpyxl") #panda 调用一个engine 叫做openpxl来帮助他读这个excel

print("原始表 shape:", raw.shape)
display(raw.iloc[:12, :8])
# ============================================================
# 2. Convert original Excel layout to modeling dataframe
#    原始格式:
#    row 0: ER015, ER016, ER017 ...
#    col 0: 变量名
#
#    目标格式:
#    每一行 = 一个 ER 批次（批次10） EP批次9
#    每一列 = 一个变量
# ============================================================

# 找到第一行里所有 ER 批次列
batch_cols = [
    c for c in raw.columns
    if isinstance(raw.iloc[0, c], str) and raw.iloc[0, c].startswith("ER")
] #[0,c] 第0行第c列 ，这个raw.column return的是列的索引

# 变量名在第一列，从 row 1 开始
# 这里排除 Grand Total
variable_rows = [
    r for r in raw.index
    if isinstance(raw.iloc[r, 0], str)
    and raw.iloc[r, 0] != "Grand Total"
] #raw.index 是行索引

variable_names = raw.loc[variable_rows, 0].astype(str).tolist() #转成python list 是一个变量名的列表

# 取数值区域，并转置
data_values = raw.loc[variable_rows, batch_cols].apply(pd.to_numeric, errors="coerce")

df = data_values.T.reset_index(drop=True) #转置.reset index是把index重置成0，1，2原来是ER001，ER002，ER003这样
df.columns = variable_names

# 加上 batch_id 插入到第0列，列名是batch id
df.insert(0, "batch_id", [raw.iloc[0, c] for c in batch_cols]) 

print("建模表 shape:", df.shape)
display(df.head())
# ============================================================
# 3. Define target y and candidate features X
# ============================================================

target_col = "熔炼炉B当前批次总气耗_PLC"

# 这个变量很可能是 target 的组成项/强泄漏变量
# 如果你的目标是分析工艺变量影响，建议先不要放进 X
leakage_col = "熔炼炉B当前批次熔炼气耗_PLC"

# 全部数值列
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# 候选特征：排除 target
candidate_features = [c for c in numeric_cols if c != target_col]

# 推荐建模特征：排除 target 和 leakage_col(可以删除)
feature_cols = [c for c in candidate_features if c != leakage_col]

print("target:", target_col)
print("候选 X 数量:", len(candidate_features))
print("推荐 X 数量，不含熔炼气耗:", len(feature_cols))
print("推荐 X:")
for c in feature_cols:
    print(" -", c)

# ============================================================
# 4. Create output folder
# ============================================================

output_dir = project_dir / "output"
os.makedirs(output_dir, exist_ok=True)
#r和f的区别 r不支持带有变量的转义 比如说:folder_name = "output"  output_dir = rf"C:\Users\fshhan17\Desktop\{folder_name}"
# ============================================================
# 5. Data quality check（缺失值）
# ============================================================

missing_summary = df[["batch_id"] + numeric_cols].isna().sum(axis=0).reset_index() #.isnan()就是如果是nan那个格子就是true不是的话就是false
missing_summary.columns = ["column", "missing_count"] #把列名改成这两个
missing_summary["missing_ratio"] = missing_summary["missing_count"] / len(df)

print("\n缺失值统计:")
display(missing_summary)

describe_summary = df[numeric_cols].describe().T #.describe()函数会自动算mean count，std，25，50，75，max
display(describe_summary)

# ============================================================
# 6. Analyze y distribution and outliers
# ============================================================

y = df[target_col]

print("\ny describe:")
display(y.describe())

# IQR 方法找 y 异常点
q1 = y.quantile(0.25)
q3 = y.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (y < lower_bound) | (y > upper_bound)
outlier_df = df.loc[outlier_mask, ["batch_id", target_col] + feature_cols].copy() #我就要batch id，target col，feature col这几列其他的都不要

print("y IQR lower bound:", lower_bound)
print("y IQR upper bound:", upper_bound)
print("y 异常点数量:", outlier_mask.sum())
display(outlier_df)

# ============================================================
# 简约版：Feature Set A/B + 3 Models
# Models: LightGBM / Linear Regression / Random Forest
# Data: only no-y-outlier data
# Output:
#   1. model_summary_df
#   2. Linear Regression equations for A/B
#   3. Excel summary
# ============================================================

import os
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lightgbm import LGBMRegressor


# ------------------------------------------------------------
# 0. Warning control
# ------------------------------------------------------------

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*X does not have valid feature names.*"
)


# ------------------------------------------------------------
# 1. Define output folders
# ------------------------------------------------------------

base_output_dir = r""

table_output_dir = os.path.join(base_output_dir, "tables")
model_output_dir = os.path.join(base_output_dir, "models")

os.makedirs(table_output_dir, exist_ok=True) #os.makedirs 这个是创建文件夹，如果他不存在的话就initial一个
os.makedirs(model_output_dir, exist_ok=True)

print("表格输出文件夹:", table_output_dir)
print("模型输出文件夹:", model_output_dir)


# ------------------------------------------------------------
# 2. Define target column
# ------------------------------------------------------------

target_col = "熔炼炉B当前批次总气耗_PLC"

if target_col not in df.columns:
    raise ValueError(f"target_col 不在 df 中，请检查列名: {target_col}")


# ------------------------------------------------------------
# 3. Define Feature Set A / B
# ------------------------------------------------------------

# A. 解释型全变量版本
feature_set_A = [
    "10#熔炼炉总投料重量(kg)",
    "10#熔炼炉固体料重量比例",
    "熔炼炉B当前批次生产时间_PLC",
    "熔炼炉B当前批次熔炼时间_PLC",###
    "熔炼炉B当前批次等待时长_PLC",
    "熔炼炉B当前批次炉门打开次数_PLC",
    "熔炼炉B当前批次炉门打开时长_PLC",
]

# B. 少共线性版本
feature_set_B = [
    "10#熔炼炉总投料重量(kg)",
    "10#熔炼炉固体料重量比例",
    "熔炼炉B当前批次熔炼时间_PLC",
    "熔炼炉B当前批次等待时长_PLC",
    "熔炼炉B当前批次炉门打开次数_PLC",
    "熔炼炉B当前批次炉门打开时长_PLC",
]

feature_sets = {
    "A_解释型全变量版本": feature_set_A,
    "B_少共线性版本": feature_set_B,
}


# ------------------------------------------------------------
# 4. Check columns
# ------------------------------------------------------------

all_required_features = sorted(set(feature_set_A + feature_set_B)) #把两个特征集合并，去重，排序 得到一个统一的特征集

missing_features = [
    col for col in all_required_features
    if col not in df.columns
]

if len(missing_features) > 0:
    raise ValueError(f"以下特征列在 df 中不存在，请检查列名: {missing_features}")

print("\nFeature Set A:")
for col in feature_set_A:
    print(" -", col)

print("\nFeature Set B:")
for col in feature_set_B:
    print(" -", col)


# ------------------------------------------------------------
# 5. Prepare no-y-outlier data
# ------------------------------------------------------------

if "outlier_mask" not in globals(): # globals() 代表：当前 Python 运行环境（全局作用域）中，所有已经定义过的变量
    raise ValueError("当前环境中没有 outlier_mask,请先运行你前面的 y outlier 检测代码。")

df_no_outlier = df.loc[~outlier_mask].copy() #~ 取反的意思

print("\n全部样本数量:", len(df))
print("y outlier 数量:", outlier_mask.sum())
print("去掉 y outlier 后样本数量:", len(df_no_outlier))


# ------------------------------------------------------------
# 6. Define only 3 models
# ------------------------------------------------------------

models = {
    "LightGBM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")), #这个意思就是用median自动补足缺失值
        ("model", LGBMRegressor(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=15,
            max_depth=4,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=2.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ]),

    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ]),
}

print("\n当前参与比较的模型:")
for model_name in models:
    print(" -", model_name)


# ============================================================
# 7. Train/test comparison for Feature Set A/B
# ============================================================

trained_models = {}
model_summary_rows = []

for feature_set_name, feature_cols in feature_sets.items():

    print("\n================================================")
    print("当前 Feature Set:", feature_set_name)
    print("================================================")

    X_current = df_no_outlier[feature_cols]
    y_current = df_no_outlier[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X_current,
        y_current,
        test_size=0.2,
        random_state=42
    )

    # 保险：强制列顺序一致
    X_train = X_train[feature_cols]
    X_test = X_test[feature_cols]

    for model_name, model in models.items():

        print("\n正在训练:", feature_set_name, "|", model_name)

        final_model = clone(model)
        final_model.fit(X_train, y_train)

        y_pred = final_model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        safe_feature_set_name = (
            feature_set_name
            .replace(" ", "_")
            .replace("/", "_")
            .replace("\\", "_")
            .replace(":", "_")
        )

        safe_model_name = (
            model_name
            .replace(" ", "_")
            .replace("/", "_")
            .replace("\\", "_")
            .replace(":", "_")
        )

        model_path = os.path.join(
            model_output_dir,
            f"{safe_feature_set_name}__{safe_model_name}_no_y_outlier.joblib" #动态生成文件名
        )

        joblib.dump(final_model, model_path) #把训练好的模型保存到硬盘中

        model_key = f"{feature_set_name}__{model_name}"
        trained_models[model_key] = final_model

        model_summary_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "RMSE_pct_of_y_test_mean": rmse / y_test.mean(),
            "R2": r2,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": len(feature_cols),
            "features": ", ".join(feature_cols),
            "model_path": model_path
        })

        print("MAE:", mae)
        print("RMSE:", rmse)
        print("RMSE / y_test_mean:", rmse / y_test.mean())
        print("R2:", r2)
        print("模型已保存到:", model_path)


model_summary_df = pd.DataFrame(model_summary_rows).sort_values(
    ["RMSE", "MAE"]
).reset_index(drop=True)

print("\n================ 模型表现对比表 ================")
display(model_summary_df)

# ============================================================
# 8. Extract Linear Regression equations for Feature Set A/B
# ============================================================

def extract_linear_equation(
    pipeline,
    feature_cols,
    feature_set_name,
    model_name="Linear Regression"
):
    """
    提取 Linear Regression 的公式参数：
    1. 标准化空间公式
    2. 原始变量单位公式

    pipeline:
        imputer -> scaler -> model
    """

    imputer = pipeline.named_steps["imputer"]
    scaler = pipeline.named_steps["scaler"]
    model = pipeline.named_steps["model"]

    coef_scaled = model.coef_
    intercept_scaled = model.intercept_

    scaler_mean = scaler.mean_
    scaler_scale = scaler.scale_

    # 原始单位下：
    # y = intercept_scaled + sum(coef_scaled_i * ((x_i - mean_i) / scale_i))
    #   = intercept_original + sum(coef_original_i * x_i)
    coef_original = coef_scaled / scaler_scale
    intercept_original = intercept_scaled - np.sum(
        coef_scaled * scaler_mean / scaler_scale
    )

    equation_df = pd.DataFrame({
        "feature_set": feature_set_name,
        "model": model_name,
        "feature": feature_cols,
        "coef_scaled_space": coef_scaled,
        "scaler_mean": scaler_mean,
        "scaler_scale": scaler_scale,
        "coef_original_unit": coef_original,
        "imputer_strategy": imputer.strategy,
        "note": "原始单位公式基于 imputer 后再 scaler 的 pipeline 换算得到"
    })

    equation_df.loc[len(equation_df)] = {
        "feature_set": feature_set_name,
        "model": model_name,
        "feature": "intercept",
        "coef_scaled_space": intercept_scaled,
        "scaler_mean": np.nan,
        "scaler_scale": np.nan,
        "coef_original_unit": intercept_original,
        "imputer_strategy": imputer.strategy,
        "note": "截距项"
    }

    print("\n================ Equation ================")
    print("Feature set:", feature_set_name)
    print("Model:", model_name)

    print("\nA. 标准化变量空间下的公式:")
    print("y = {:.6f}".format(intercept_scaled))

    for feature, coef in zip(feature_cols, coef_scaled):
        print("    + ({:.6f}) * standardized({})".format(coef, feature))

    print("\nB. 原始变量单位下的公式:")
    print("y = {:.6f}".format(intercept_original))

    for feature, coef in zip(feature_cols, coef_original):
        print("    + ({:.6f}) * {}".format(coef, feature))

    return equation_df


linear_equation_list = []

for feature_set_name, feature_cols in feature_sets.items():

    model_key = f"{feature_set_name}__Linear Regression"
    fitted_linear_pipeline = trained_models[model_key]

    equation_df = extract_linear_equation(
        pipeline=fitted_linear_pipeline,
        feature_cols=feature_cols,
        feature_set_name=feature_set_name
    )

    linear_equation_list.append(equation_df)


linear_equation_all_df = pd.concat(
    linear_equation_list,
    ignore_index=True
)

print("\n================ Linear Regression 参数表 ================")
display(linear_equation_all_df)

# ============================================================
# 9. Save# 9. Save simplified output to Excel

simple_output_path = os.path.join(
    table_output_dir,
    "simple_model_comparison_AB_no_y_outlier.xlsx"
)

with pd.ExcelWriter(simple_output_path, engine="openpyxl") as writer:
    model_summary_df.to_excel(
        writer,
        sheet_name="model_comparison",
        index=False
    )

    linear_equation_all_df.to_excel(
        writer,
        sheet_name="linear_equations",
        index=False
    )

    pd.DataFrame({
        "feature_set": "A_解释型全变量版本",
        "feature": feature_set_A
    }).to_excel(
        writer,
        sheet_name="feature_set_A",
        index=False
    )

    pd.DataFrame({
        "feature_set": "B_少共线性版本",
        "feature": feature_set_B
    }).to_excel(
        writer,
        sheet_name="feature_set_B",
        index=False
    )

print("\n简约版结果已保存到:", simple_output_path)



ModuleNotFoundError: No module named 'sklearn'

测试不同搜索最优参数算法

In [2]:
# ============================================================
# Recommendation System V2:
# Historical + Random Forest + Linear Regression + LightGBM
#
# Given total charging weight, recommend controllable variables
#
# Goal:
#   根据工厂输入的总投料重量，推荐 4 个可控变量：
#   1. 10#熔炼炉固体料重量比例
#   2. 熔炼炉B当前批次等待时长_PLC
#   3. 熔炼炉B当前批次炉门打开次数_PLC
#   4. 熔炼炉B当前批次炉门打开时长_PLC
#
# Important:
#   推荐系统优先使用 B_少共线性版本模型。
#   不建议使用包含 生产时间_PLC 的 A 版本模型。
#
# Methods:
#   1. Historical similar low-gas benchmark
#   2. Random Forest constrained optimization
#   3. Linear Regression constrained optimization
#   4. LightGBM constrained optimization
#   5. Fused recommendation
# ============================================================

import os
import glob
import joblib
import numpy as np
import pandas as pd


# ============================================================
# 0. Define paths
# ============================================================

base_output_dir = project_dir

table_output_dir = os.path.join(base_output_dir, "tables")
figure_output_dir = os.path.join(base_output_dir, "figures")
model_output_dir = os.path.join(base_output_dir, "models")
recommend_output_dir = os.path.join(base_output_dir, "recommendation")

os.makedirs(recommend_output_dir, exist_ok=True)

print("模型 checkpoint 文件夹:", model_output_dir)
print("推荐结果输出文件夹:", recommend_output_dir)


# ============================================================
# 1. Define columns
# ============================================================

target_col = "熔炼炉B当前批次总气耗_PLC"

weight_col = "10#熔炼炉总投料重量(kg)"

# 真正推荐给工厂的 4 个可控变量
controllable_cols = [
    "10#熔炼炉固体料重量比例",
    "熔炼炉B当前批次等待时长_PLC",
    "熔炼炉B当前批次炉门打开次数_PLC",
    "熔炼炉B当前批次炉门打开时长_PLC",
]

# 不作为推荐变量，但是模型如果需要，会自动用历史参考值补上
melting_time_col = "熔炼炉B当前批次熔炼时间_PLC"
production_time_col = "熔炼炉B当前批次生产时间_PLC"

# 推荐系统优先使用这个 feature set
preferred_feature_set_keyword = "B_少共线性版本"


# ============================================================
# 2. Helper:
#    Safe display for script / notebook
# ============================================================

def safe_display(obj):
    """
    兼容 Jupyter Notebook 和普通 .py 脚本。
    Notebook 中使用 display；
    普通 Python 脚本中使用 print。
    """
    try:
        display(obj)
    except NameError:
        print(obj)


# ============================================================
# 3. Helper:
#    Get model feature names
# ============================================================

def get_model_feature_names(trained_model):
    """
    自动读取 sklearn Pipeline / model 训练时使用的特征名。

    目的：
    - predict 时必须传入和 fit 时完全一致的列名和列顺序
    - 避免手写 feature_cols 时漏列或顺序错乱
    """

    if hasattr(trained_model, "feature_names_in_"):
        return list(trained_model.feature_names_in_)

    if hasattr(trained_model, "named_steps"):
        for step_name, step in trained_model.named_steps.items():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise ValueError(
        "无法从模型 checkpoint 中读取 feature_names_in_。"
        "请确认模型训练时输入的是 pandas DataFrame，而不是 numpy array。"
    )


# ============================================================
# 4. Helper:
#    Find and load B-version model checkpoint safely
# ============================================================

def find_and_load_b_version_model(
    model_output_dir,
    model_keywords,
    preferred_feature_set_keyword="B_少共线性版本",
    forbidden_cols=None
):
    """
    安全读取推荐系统要用的模型 checkpoint。

    支持单个关键词或多个关键词。

    逻辑：
    1. 找到所有包含 model_keywords 的 joblib 文件
    2. 优先选择文件名中包含 B_少共线性版本 的模型
    3. 自动读取模型训练特征
    4. 排除包含 forbidden_cols 的模型，例如 production_time_col
    5. 如果没有 B 文件名，但某个模型特征也不含 forbidden_cols，也允许作为 fallback
    6. 如果所有模型都包含 forbidden_cols，直接报错，避免误用 A 版本
    """

    if forbidden_cols is None:
        forbidden_cols = []

    if isinstance(model_keywords, str):
        model_keywords = [model_keywords]

    files = []

    for keyword in model_keywords:
        pattern = os.path.join(model_output_dir, f"*{keyword}*.joblib")
        matched_files = glob.glob(pattern)
        files.extend(matched_files)

    files = sorted(list(set(files)))

    if len(files) == 0:
        raise FileNotFoundError(
            f"没有在 {model_output_dir} 找到包含关键词 {model_keywords} 的 joblib 文件。"
        )

    candidate_infos = []

    print(f"\n正在查找模型关键词: {model_keywords}")
    print("找到候选 checkpoint 数量:", len(files))

    for file_path in files:
        try:
            model = joblib.load(file_path)
            feature_cols = get_model_feature_names(model)

            has_forbidden_col = any(col in feature_cols for col in forbidden_cols)
            is_preferred_b = preferred_feature_set_keyword in os.path.basename(file_path)

            candidate_infos.append({
                "file_path": file_path,
                "model": model,
                "feature_cols": feature_cols,
                "has_forbidden_col": has_forbidden_col,
                "is_preferred_b": is_preferred_b,
                "n_features": len(feature_cols),
            })

            print("\n候选模型:", file_path)
            print("是否文件名包含 B_少共线性版本:", is_preferred_b)
            print("是否包含 forbidden cols:", has_forbidden_col)
            print("训练特征:")
            for c in feature_cols:
                print(" -", c)

        except Exception as e:
            print("\n读取模型失败:", file_path)
            print("错误:", e)

    if len(candidate_infos) == 0:
        raise ValueError(f"所有 {model_keywords} checkpoint 都读取失败。")

    # ------------------------------------------------------------
    # 第一优先级：文件名包含 B，而且不含 forbidden cols
    # ------------------------------------------------------------

    preferred_candidates = [
        info for info in candidate_infos
        if info["is_preferred_b"] and not info["has_forbidden_col"]
    ]

    if len(preferred_candidates) > 0:
        selected = preferred_candidates[0]
        print(f"\n最终选择模型: B 版本 checkpoint")
        print("模型路径:", selected["file_path"])
        return selected["model"], selected["file_path"], selected["feature_cols"]

    # ------------------------------------------------------------
    # 第二优先级：文件名不一定包含 B，但是特征不含 forbidden cols
    # ------------------------------------------------------------

    fallback_candidates = [
        info for info in candidate_infos
        if not info["has_forbidden_col"]
    ]

    if len(fallback_candidates) > 0:
        selected = fallback_candidates[0]
        print(f"\n警告: 没有找到文件名包含 B 的模型，但找到了不含 forbidden cols 的模型。")
        print("将使用该模型作为 fallback。")
        print("模型路径:", selected["file_path"])
        return selected["model"], selected["file_path"], selected["feature_cols"]

    # ------------------------------------------------------------
    # 如果所有模型都包含 production_time_col，说明只找到了 A 版本
    # ------------------------------------------------------------

    raise ValueError(
        f"找到的模型全部包含 forbidden columns: {forbidden_cols}。\n"
        f"这很可能是 A_解释型全变量版本，不建议用于推荐系统。\n"
        f"请先训练并保存 B_少共线性版本模型。"
    )


# ============================================================
# 5. Load model checkpoints:
#    Random Forest + Linear Regression + LightGBM
# ============================================================

rf_model, rf_model_path, rf_feature_cols = find_and_load_b_version_model(
    model_output_dir=model_output_dir,
    model_keywords="Random_Forest",
    preferred_feature_set_keyword=preferred_feature_set_keyword,
    forbidden_cols=[production_time_col]
)

lr_model, lr_model_path, lr_feature_cols = find_and_load_b_version_model(
    model_output_dir=model_output_dir,
    model_keywords=["Linear_Regression", "Ridge_Regression"],
    preferred_feature_set_keyword=preferred_feature_set_keyword,
    forbidden_cols=[production_time_col]
)

lgbm_model, lgbm_model_path, lgbm_feature_cols = find_and_load_b_version_model(
    model_output_dir=model_output_dir,
    model_keywords="LightGBM",
    preferred_feature_set_keyword=preferred_feature_set_keyword,
    forbidden_cols=[production_time_col]
)

print("\n================ 最终使用的模型 checkpoint ================")
print("Random Forest:", rf_model_path)
print("Linear Regression:", lr_model_path)
print("LightGBM:", lgbm_model_path)

print("\nRandom Forest 训练特征:")
for c in rf_feature_cols:
    print(" -", c)

print("\nLinear Regression 训练特征:")
for c in lr_feature_cols:
    print(" -", c)

print("\nLightGBM 训练特征:")
for c in lgbm_feature_cols:
    print(" -", c)


# ============================================================
# 6. Basic column checks
#    这里假设你前面已经有 df_no_outlier
# ============================================================

required_cols = (
    [target_col, weight_col]
    + controllable_cols
    + list(set(rf_feature_cols + lr_feature_cols + lgbm_feature_cols))
)

required_cols = list(dict.fromkeys(required_cols))

missing_cols = [c for c in required_cols if c not in df_no_outlier.columns]

if len(missing_cols) > 0:
    raise ValueError(f"df_no_outlier 中缺少以下列，请检查列名: {missing_cols}")

print("\n需要的列都存在，可以继续。")


# ============================================================
# 7. Method 1:
#    Historical similar batch benchmark recommendation
# ============================================================

def historical_benchmark_recommendation(
    df_reference,
    total_weight,
    weight_col,
    target_col,
    controllable_cols,
    tolerance_ratio=0.05,
    low_gas_top_ratio=0.2
):
    """
    给定总投料量：

    1. 找总投料量相似的历史批次
    2. 按真实总气耗从低到高排序
    3. 取低气耗前 low_gas_top_ratio
    4. 输出 4 个可控变量的历史 benchmark 推荐范围
    """

    low_weight = total_weight * (1 - tolerance_ratio)
    high_weight = total_weight * (1 + tolerance_ratio)

    similar_df = df_reference[
        (df_reference[weight_col] >= low_weight)
        & (df_reference[weight_col] <= high_weight)
    ].copy()

    if len(similar_df) == 0:
        raise ValueError(
            f"没有找到总投料量在 {low_weight:.1f} 到 {high_weight:.1f} 之间的历史批次。"
        )

    similar_df = similar_df.sort_values(target_col, ascending=True)

    n_top = max(1, int(np.ceil(len(similar_df) * low_gas_top_ratio)))

    low_gas_df = similar_df.head(n_top).copy()

    summary_rows = []

    summary_cols = controllable_cols + [target_col]

    for col in summary_cols:
        summary_rows.append({
            "variable": col,
            "method": "historical_benchmark",
            "value_median": low_gas_df[col].median(),
            "value_mean": low_gas_df[col].mean(),
            "value_p10": low_gas_df[col].quantile(0.10),
            "value_p25": low_gas_df[col].quantile(0.25),
            "value_p75": low_gas_df[col].quantile(0.75),
            "value_p90": low_gas_df[col].quantile(0.90),
            "value_min": low_gas_df[col].min(),
            "value_max": low_gas_df[col].max(),
            "n_similar_batches": len(similar_df),
            "n_low_gas_batches": len(low_gas_df),
            "weight_low_bound": low_weight,
            "weight_high_bound": high_weight,
        })

    benchmark_summary = pd.DataFrame(summary_rows)

    return similar_df, low_gas_df, benchmark_summary


# ============================================================
# 8. Method 2:
#    Model-based constrained optimization recommendation
# ============================================================

def model_based_optimization_recommendation(
    trained_model,
    df_reference,
    similar_df,
    low_gas_df,
    total_weight,
    feature_cols,
    weight_col,
    target_col,
    controllable_cols,
    n_candidates=50000,
    top_n=50,
    random_state=42,
    range_low_q=0.05,
    range_high_q=0.95,
    range_source="similar",
    fixed_value_strategy="low_gas_median"
):
    """
    模型约束优化推荐。

    固定：
        - 总投料重量 = 工厂输入 total_weight

    优化：
        - 10#熔炼炉固体料重量比例
        - 熔炼炉B当前批次等待时长_PLC
        - 熔炼炉B当前批次炉门打开次数_PLC
        - 熔炼炉B当前批次炉门打开时长_PLC

    不推荐但自动补值：
        - 熔炼炉B当前批次熔炼时间_PLC
        - 其他模型训练时用过但不在 controllable_cols 里的变量
    """

    rng = np.random.default_rng(random_state)

    # ------------------------------------------------------------
    # 1. Decide random search range source
    # ------------------------------------------------------------

    if range_source == "similar":
        range_base_df = similar_df.copy()
    elif range_source == "all":
        range_base_df = df_reference.copy()
    elif range_source == "low_gas":
        range_base_df = low_gas_df.copy()
    else:
        raise ValueError("range_source 只能是 'similar', 'all', 或 'low_gas'")

    # ------------------------------------------------------------
    # 2. Define historical search ranges
    # ------------------------------------------------------------

    range_rows = []
    ranges = {}

    for col in controllable_cols:
        low = range_base_df[col].quantile(range_low_q)
        high = range_base_df[col].quantile(range_high_q)

        if pd.isna(low) or pd.isna(high):
            raise ValueError(f"{col} 的搜索范围存在 NaN，请检查该列数据。")

        if low == high:
            print(f"警告: {col} 的 search_low 和 search_high 相同，候选值将固定为 {low}")

        ranges[col] = (low, high)

        range_rows.append({
            "variable": col,
            "range_source": range_source,
            "range_low_q": range_low_q,
            "range_high_q": range_high_q,
            "search_low": low,
            "search_high": high,
        })

    range_df = pd.DataFrame(range_rows)

    # ------------------------------------------------------------
    # 3. Generate candidate combinations
    # ------------------------------------------------------------

    candidates = pd.DataFrame()

    # 固定工厂输入总投料重量
    candidates[weight_col] = np.repeat(total_weight, n_candidates)

    # 只对 4 个可控变量做 random search
    for col in controllable_cols:
        low, high = ranges[col]

        if "次数" in col:
            low_int = int(np.floor(low))
            high_int = int(np.ceil(high))

            if low_int == high_int:
                candidates[col] = np.repeat(low_int, n_candidates)
            else:
                candidates[col] = rng.integers(
                    low=low_int,
                    high=high_int + 1,
                    size=n_candidates
                )
        else:
            if low == high:
                candidates[col] = np.repeat(low, n_candidates)
            else:
                candidates[col] = rng.uniform(
                    low=low,
                    high=high,
                    size=n_candidates
                )

    # ------------------------------------------------------------
    # 4. Automatically fill model-required but not recommended variables
    # ------------------------------------------------------------

    fixed_feature_rows = []

    for col in feature_cols:

        if col in candidates.columns:
            continue

        if fixed_value_strategy == "low_gas_median":
            fixed_value = low_gas_df[col].median()
            fixed_source = "low_gas_median"

        elif fixed_value_strategy == "similar_median":
            fixed_value = similar_df[col].median()
            fixed_source = "similar_median"

        else:
            raise ValueError(
                "fixed_value_strategy 只能是 'low_gas_median' 或 'similar_median'"
            )

        if pd.isna(fixed_value):
            raise ValueError(
                f"模型需要变量 {col}，但是无法计算固定参考值。"
                f"请检查该列在 low_gas_df / similar_df 中是否全是 NaN。"
            )

        candidates[col] = np.repeat(fixed_value, n_candidates)

        fixed_feature_rows.append({
            "variable": col,
            "fixed_value": fixed_value,
            "fixed_source": fixed_source,
            "reason": "model_required_but_not_recommended",
        })

    fixed_feature_df = pd.DataFrame(fixed_feature_rows)

    # ------------------------------------------------------------
    # 5. Ensure exact feature names and order
    # ------------------------------------------------------------

    missing_candidate_cols = [
        c for c in feature_cols
        if c not in candidates.columns
    ]

    if len(missing_candidate_cols) > 0:
        raise ValueError(
            f"候选数据中仍然缺少模型需要的特征列: {missing_candidate_cols}"
        )

    candidate_X = candidates[feature_cols].copy()

    # ------------------------------------------------------------
    # 6. Predict gas consumption
    # ------------------------------------------------------------

    candidates["predicted_gas"] = trained_model.predict(candidate_X)

    # ------------------------------------------------------------
    # 7. Select Top N lowest predicted gas candidates
    # ------------------------------------------------------------

    top_candidates = candidates.sort_values(
        "predicted_gas",
        ascending=True
    ).head(top_n).copy()

    # ------------------------------------------------------------
    # 8. Recommendation summary
    # ------------------------------------------------------------

    summary_rows = []

    fixed_cols = [
        c for c in feature_cols
        if c not in controllable_cols and c != weight_col
    ]

    summary_cols = controllable_cols + fixed_cols + ["predicted_gas"]

    for col in summary_cols:
        summary_rows.append({
            "variable": col,
            "method": "model_optimization",
            "value_median": top_candidates[col].median(),
            "value_mean": top_candidates[col].mean(),
            "value_p10": top_candidates[col].quantile(0.10),
            "value_p25": top_candidates[col].quantile(0.25),
            "value_p75": top_candidates[col].quantile(0.75),
            "value_p90": top_candidates[col].quantile(0.90),
            "value_min": top_candidates[col].min(),
            "value_max": top_candidates[col].max(),
            "n_candidates": n_candidates,
            "top_n": top_n,
        })

    recommendation_summary = pd.DataFrame(summary_rows)

    return (
        candidates,
        top_candidates,
        recommendation_summary,
        range_df,
        fixed_feature_df
    )


# ============================================================
# 9. Validate model recommendation against historical benchmark
# ============================================================

def validate_recommendation_against_benchmark(
    model_summary,
    benchmark_summary,
    controllable_cols
):
    """
    检查模型推荐的 4 个可控变量中位数，
    是否落在历史相似低气耗批次 p10~p90 区间内。
    """

    rows = []

    for col in controllable_cols:
        model_row = model_summary[
            model_summary["variable"] == col
        ].iloc[0]

        bench_row = benchmark_summary[
            benchmark_summary["variable"] == col
        ].iloc[0]

        model_median = model_row["value_median"]
        bench_p10 = bench_row["value_p10"]
        bench_p90 = bench_row["value_p90"]
        bench_median = bench_row["value_median"]

        in_benchmark_range = (
            model_median >= bench_p10
            and model_median <= bench_p90
        )

        rows.append({
            "variable": col,
            "model_recommended_median": model_median,
            "benchmark_median": bench_median,
            "benchmark_p10": bench_p10,
            "benchmark_p90": bench_p90,
            "model_median_in_benchmark_p10_p90": in_benchmark_range,
            "difference_vs_benchmark_median": model_median - bench_median,
        })

    return pd.DataFrame(rows)


# ============================================================
# 10. Build factory-facing final recommendation table
# ============================================================

def build_factory_recommendation_table(
    combined_summary,
    controllable_cols
):
    """
    只给工厂展示真正要推荐的 4 个变量。
    不展示熔炼时间 / 生产时间作为推荐变量。
    """

    factory_table = combined_summary[
        combined_summary["variable"].isin(controllable_cols)
    ][
        [
            "variable",
            "model",
            "value_median",
            "value_p10",
            "value_p25",
            "value_p75",
            "value_p90",
            "value_min",
            "value_max",
        ]
    ].copy()

    factory_table = factory_table.rename(columns={
        "variable": "推荐变量",
        "model": "推荐方法",
        "value_median": "推荐值_中位数",
        "value_p10": "推荐范围_p10",
        "value_p25": "推荐范围_p25",
        "value_p75": "推荐范围_p75",
        "value_p90": "推荐范围_p90",
        "value_min": "Top候选最小值",
        "value_max": "Top候选最大值",
    })

    return factory_table


# ============================================================
# 11. Fuse RF / Linear Regression / LightGBM / Historical benchmark
# ============================================================

def build_fused_recommendation_table(
    combined_summary,
    validation_df,
    controllable_cols
):
    """
    三模型融合推荐逻辑：

    对每个可控变量：
    1. 先检查 RF / Linear Regression / LightGBM 的推荐中位数是否落在
       historical benchmark p10~p90 内。

    2. 如果有模型在 benchmark 范围内：
       只融合这些 valid models。

       默认基础权重：
       - Random Forest:      0.40
       - LightGBM:           0.40
       - Linear Regression:  0.20

       如果某些模型 invalid，就把剩下 valid models 的权重重新归一化。

    3. 如果三个模型都不在 benchmark 范围内：
       直接使用 Historical Benchmark median。

    4. 最后保险：
       fused value clip 到 historical benchmark p10~p90。
    """

    base_weights = {
        "Random Forest": 0.40,
        "LightGBM": 0.40,
        "Linear Regression": 0.20,
    }

    model_names = ["Random Forest", "Linear Regression", "LightGBM"]

    rows = []

    for col in controllable_cols:

        hist_row = combined_summary[
            (combined_summary["variable"] == col)
            & (combined_summary["model"] == "Historical Benchmark")
        ].iloc[0]

        hist_value = hist_row["value_median"]
        benchmark_p10 = hist_row["value_p10"]
        benchmark_p90 = hist_row["value_p90"]

        model_values = {}
        model_valid_flags = {}

        for model_name in model_names:

            model_row = combined_summary[
                (combined_summary["variable"] == col)
                & (combined_summary["model"] == model_name)
            ].iloc[0]

            model_value = model_row["value_median"]
            model_values[model_name] = model_value

            valid_flag = validation_df[
                (validation_df["variable"] == col)
                & (validation_df["model"] == model_name)
            ]["model_median_in_benchmark_p10_p90"].iloc[0]

            model_valid_flags[model_name] = valid_flag

        valid_models = [
            model_name for model_name in model_names
            if model_valid_flags[model_name]
        ]

        if len(valid_models) > 0:

            weight_sum = sum(base_weights[m] for m in valid_models)

            normalized_weights = {
                m: base_weights[m] / weight_sum
                for m in valid_models
            }

            fused_value_raw = sum(
                normalized_weights[m] * model_values[m]
                for m in valid_models
            )

            fusion_rule = (
                "Weighted average of valid models: "
                + ", ".join([
                    f"{m} weight={normalized_weights[m]:.2f}"
                    for m in valid_models
                ])
            )

        else:
            fused_value_raw = hist_value
            fusion_rule = (
                "All models outside benchmark range, "
                "use Historical Benchmark median"
            )

        fused_value_clipped = np.clip(
            fused_value_raw,
            benchmark_p10,
            benchmark_p90
        )

        was_clipped = fused_value_raw != fused_value_clipped

        rows.append({
            "推荐变量": col,

            "Historical_Benchmark_Median": hist_value,

            "Random_Forest_Median": model_values["Random Forest"],
            "Linear_Regression_Median": model_values["Linear Regression"],
            "LightGBM_Median": model_values["LightGBM"],

            "RF_in_Benchmark_Range": model_valid_flags["Random Forest"],
            "Linear_Regression_in_Benchmark_Range": model_valid_flags["Linear Regression"],
            "LightGBM_in_Benchmark_Range": model_valid_flags["LightGBM"],

            "Benchmark_p10": benchmark_p10,
            "Benchmark_p90": benchmark_p90,

            "Fusion_Rule": fusion_rule,
            "Fused_Recommendation_Raw": fused_value_raw,
            "Fused_Recommendation_Final": fused_value_clipped,
            "was_clipped": was_clipped,
        })

    fused_df = pd.DataFrame(rows)

    return fused_df


# ============================================================
# 12. Main function:
#     Compare benchmark vs RF vs Linear Regression vs LightGBM
# ============================================================

def run_recommendation_comparison(
    total_weight,
    df_reference,
    rf_model,
    lr_model,
    lgbm_model,
    rf_feature_cols,
    lr_feature_cols,
    lgbm_feature_cols,
    weight_col,
    target_col,
    controllable_cols,
    tolerance_ratio=0.05,
    low_gas_top_ratio=0.2,
    n_candidates=50000,
    top_n=50,
    random_state=42,
    range_source="similar",
    fixed_value_strategy="low_gas_median"
):
    """
    对给定总投料量，比较：

    1. Historical Similar All Batches
    2. Historical Benchmark Top Low Gas
    3. Random Forest Optimization
    4. Linear Regression Optimization
    5. LightGBM Optimization

    并输出：
    - 工厂真正需要看的 4 个变量推荐
    - RF / Linear Regression / LightGBM / Benchmark 对比
    - 模型推荐是否落在 benchmark p10~p90
    - fixed variables 的使用值
    - 融合推荐
    """

    # ------------------------------------------------------------
    # Method 1: historical benchmark
    # ------------------------------------------------------------

    similar_df, low_gas_df, benchmark_summary = historical_benchmark_recommendation(
        df_reference=df_reference,
        total_weight=total_weight,
        weight_col=weight_col,
        target_col=target_col,
        controllable_cols=controllable_cols,
        tolerance_ratio=tolerance_ratio,
        low_gas_top_ratio=low_gas_top_ratio
    )

    print("\n相似批次数量:", len(similar_df))
    print("低气耗 benchmark 批次数量:", len(low_gas_df))

    # ------------------------------------------------------------
    # Method 2A: Random Forest optimization
    # ------------------------------------------------------------

    (
        rf_candidates,
        rf_top,
        rf_summary,
        search_range_df,
        rf_fixed_feature_df
    ) = model_based_optimization_recommendation(
        trained_model=rf_model,
        df_reference=df_reference,
        similar_df=similar_df,
        low_gas_df=low_gas_df,
        total_weight=total_weight,
        feature_cols=rf_feature_cols,
        weight_col=weight_col,
        target_col=target_col,
        controllable_cols=controllable_cols,
        n_candidates=n_candidates,
        top_n=top_n,
        random_state=random_state,
        range_source=range_source,
        fixed_value_strategy=fixed_value_strategy
    )

    rf_summary["model"] = "Random Forest"

    # ------------------------------------------------------------
    # Method 2B: Linear Regression optimization
    # ------------------------------------------------------------

    (
        lr_candidates,
        lr_top,
        lr_summary,
        _,
        lr_fixed_feature_df
    ) = model_based_optimization_recommendation(
        trained_model=lr_model,
        df_reference=df_reference,
        similar_df=similar_df,
        low_gas_df=low_gas_df,
        total_weight=total_weight,
        feature_cols=lr_feature_cols,
        weight_col=weight_col,
        target_col=target_col,
        controllable_cols=controllable_cols,
        n_candidates=n_candidates,
        top_n=top_n,
        random_state=random_state + 1,
        range_source=range_source,
        fixed_value_strategy=fixed_value_strategy
    )

    lr_summary["model"] = "Linear Regression"

    # ------------------------------------------------------------
    # Method 2C: LightGBM optimization
    # ------------------------------------------------------------

    (
        lgbm_candidates,
        lgbm_top,
        lgbm_summary,
        _,
        lgbm_fixed_feature_df
    ) = model_based_optimization_recommendation(
        trained_model=lgbm_model,
        df_reference=df_reference,
        similar_df=similar_df,
        low_gas_df=low_gas_df,
        total_weight=total_weight,
        feature_cols=lgbm_feature_cols,
        weight_col=weight_col,
        target_col=target_col,
        controllable_cols=controllable_cols,
        n_candidates=n_candidates,
        top_n=top_n,
        random_state=random_state + 2,
        range_source=range_source,
        fixed_value_strategy=fixed_value_strategy
    )

    lgbm_summary["model"] = "LightGBM"

    # ------------------------------------------------------------
    # Combined summary
    # ------------------------------------------------------------

    benchmark_summary_for_compare = benchmark_summary.copy()
    benchmark_summary_for_compare["model"] = "Historical Benchmark"

    combined_summary = pd.concat(
        [
            benchmark_summary_for_compare,
            rf_summary,
            lr_summary,
            lgbm_summary,
        ],
        ignore_index=True
    )

    # ------------------------------------------------------------
    # Validation against benchmark
    # ------------------------------------------------------------

    rf_validation = validate_recommendation_against_benchmark(
        model_summary=rf_summary,
        benchmark_summary=benchmark_summary,
        controllable_cols=controllable_cols
    )
    rf_validation["model"] = "Random Forest"

    lr_validation = validate_recommendation_against_benchmark(
        model_summary=lr_summary,
        benchmark_summary=benchmark_summary,
        controllable_cols=controllable_cols
    )
    lr_validation["model"] = "Linear Regression"

    lgbm_validation = validate_recommendation_against_benchmark(
        model_summary=lgbm_summary,
        benchmark_summary=benchmark_summary,
        controllable_cols=controllable_cols
    )
    lgbm_validation["model"] = "LightGBM"

    validation_df = pd.concat(
        [rf_validation, lr_validation, lgbm_validation],
        ignore_index=True
    )

    # ------------------------------------------------------------
    # Method-level gas comparison
    # ------------------------------------------------------------

    method_score_rows = []

    method_score_rows.append({
        "method": "Historical Similar All Batches",
        "gas_metric_type": "actual_all_similar_batches",
        "gas_median": similar_df[target_col].median(),
        "gas_mean": similar_df[target_col].mean(),
        "gas_p10": similar_df[target_col].quantile(0.10),
        "gas_p90": similar_df[target_col].quantile(0.90),
        "n_records": len(similar_df),
    })

    method_score_rows.append({
        "method": "Historical Benchmark Top Low Gas",
        "gas_metric_type": "actual_low_gas_batches",
        "gas_median": low_gas_df[target_col].median(),
        "gas_mean": low_gas_df[target_col].mean(),
        "gas_p10": low_gas_df[target_col].quantile(0.10),
        "gas_p90": low_gas_df[target_col].quantile(0.90),
        "n_records": len(low_gas_df),
    })

    method_score_rows.append({
        "method": "Random Forest Optimization",
        "gas_metric_type": "predicted_top_candidates",
        "gas_median": rf_top["predicted_gas"].median(),
        "gas_mean": rf_top["predicted_gas"].mean(),
        "gas_p10": rf_top["predicted_gas"].quantile(0.10),
        "gas_p90": rf_top["predicted_gas"].quantile(0.90),
        "n_records": len(rf_top),
    })

    method_score_rows.append({
        "method": "Linear Regression Optimization",
        "gas_metric_type": "predicted_top_candidates",
        "gas_median": lr_top["predicted_gas"].median(),
        "gas_mean": lr_top["predicted_gas"].mean(),
        "gas_p10": lr_top["predicted_gas"].quantile(0.10),
        "gas_p90": lr_top["predicted_gas"].quantile(0.90),
        "n_records": len(lr_top),
    })

    method_score_rows.append({
        "method": "LightGBM Optimization",
        "gas_metric_type": "predicted_top_candidates",
        "gas_median": lgbm_top["predicted_gas"].median(),
        "gas_mean": lgbm_top["predicted_gas"].mean(),
        "gas_p10": lgbm_top["predicted_gas"].quantile(0.10),
        "gas_p90": lgbm_top["predicted_gas"].quantile(0.90),
        "n_records": len(lgbm_top),
    })

    method_score_df = pd.DataFrame(method_score_rows)

    # ------------------------------------------------------------
    # Improvement vs all similar batches
    # ------------------------------------------------------------

    baseline_gas_median = method_score_df.loc[
        method_score_df["method"] == "Historical Similar All Batches",
        "gas_median"
    ].iloc[0]

    baseline_gas_mean = method_score_df.loc[
        method_score_df["method"] == "Historical Similar All Batches",
        "gas_mean"
    ].iloc[0]

    method_score_df["improvement_vs_all_similar_median_%"] = (
        baseline_gas_median - method_score_df["gas_median"]
    ) / baseline_gas_median * 100

    method_score_df["improvement_vs_all_similar_mean_%"] = (
        baseline_gas_mean - method_score_df["gas_mean"]
    ) / baseline_gas_mean * 100

    # ------------------------------------------------------------
    # Factory-facing recommendation table
    # ------------------------------------------------------------

    factory_recommendation_table = build_factory_recommendation_table(
        combined_summary=combined_summary,
        controllable_cols=controllable_cols
    )

    # ------------------------------------------------------------
    # Fused recommendation table
    # ------------------------------------------------------------

    fused_recommendation_table = build_fused_recommendation_table(
        combined_summary=combined_summary,
        validation_df=validation_df,
        controllable_cols=controllable_cols
    )

    return {
        "similar_batches": similar_df,
        "low_gas_batches": low_gas_df,
        "benchmark_summary": benchmark_summary,
        "search_range": search_range_df,

        "rf_top_candidates": rf_top,
        "lr_top_candidates": lr_top,
        "lgbm_top_candidates": lgbm_top,

        "combined_summary": combined_summary,
        "validation": validation_df,
        "method_score": method_score_df,
        "factory_recommendation": factory_recommendation_table,
        "fused_recommendation": fused_recommendation_table,

        "rf_fixed_features": rf_fixed_feature_df,
        "lr_fixed_features": lr_fixed_feature_df,
        "lgbm_fixed_features": lgbm_fixed_feature_df,
    }


# ============================================================
# 13. Example call
#     工厂输入一个总投料量，例如 86000 kg
# ============================================================

factory_input_total_weight = 86000

result = run_recommendation_comparison(
    total_weight=factory_input_total_weight,
    df_reference=df_no_outlier,

    rf_model=rf_model,
    lr_model=lr_model,
    lgbm_model=lgbm_model,

    rf_feature_cols=rf_feature_cols,
    lr_feature_cols=lr_feature_cols,
    lgbm_feature_cols=lgbm_feature_cols,

    weight_col=weight_col,
    target_col=target_col,
    controllable_cols=controllable_cols,

    tolerance_ratio=0.05,                  # 相似批次: 总投料量 ±5%
    low_gas_top_ratio=0.2,                 # 历史低气耗: 前 20%
    n_candidates=50000,                    # random search 候选数
    top_n=50,                              # 取预测气耗最低 Top 50
    random_state=42,
    range_source="similar",                # 推荐使用相似批次范围
    fixed_value_strategy="low_gas_median"  # 非推荐变量用低气耗相似批次 median
)


# ============================================================
# 14. Display key outputs
# ============================================================

print("\n================ 方法整体气耗对比 ================")
safe_display(result["method_score"])

print("\n================ 给工厂看的推荐结果：只包含 4 个可控变量 ================")
safe_display(result["factory_recommendation"])

print("\n================ 融合后的最终推荐结果 ================")
safe_display(result["fused_recommendation"])

print("\n================ 四种方法推荐变量汇总 ================")
safe_display(
    result["combined_summary"][
        result["combined_summary"]["variable"].isin(
            controllable_cols + ["predicted_gas", target_col]
        )
    ].sort_values(["variable", "model"])
)

print("\n================ 模型推荐是否落在历史低气耗范围内 ================")
safe_display(result["validation"])

print("\n================ Random Forest 自动固定的非推荐变量 ================")
safe_display(result["rf_fixed_features"])

print("\n================ Linear Regression 自动固定的非推荐变量 ================")
safe_display(result["lr_fixed_features"])

print("\n================ LightGBM 自动固定的非推荐变量 ================")
safe_display(result["lgbm_fixed_features"])

print("\n================ Random Forest Top Candidates ================")
safe_display(result["rf_top_candidates"].head(10))

print("\n================ Linear Regression Top Candidates ================")
safe_display(result["lr_top_candidates"].head(10))

print("\n================ LightGBM Top Candidates ================")
safe_display(result["lgbm_top_candidates"].head(10))

print("\n================ 历史相似低气耗批次 ================")
safe_display(result["low_gas_batches"].head(10))


# ============================================================
# 15. Export recommendation results to Excel
# ============================================================

recommend_excel_path = os.path.join(
    recommend_output_dir,
    f"recommendation_total_weight_{int(factory_input_total_weight)}_with_lightgbm.xlsx"
)

with pd.ExcelWriter(recommend_excel_path, engine="openpyxl") as writer:

    result["method_score"].to_excel(
        writer,
        sheet_name="method_score",
        index=False
    )

    result["factory_recommendation"].to_excel(
        writer,
        sheet_name="factory_recommendation",
        index=False
    )

    result["fused_recommendation"].to_excel(
        writer,
        sheet_name="fused_recommendation",
        index=False
    )

    result["combined_summary"].to_excel(
        writer,
        sheet_name="combined_summary",
        index=False
    )

    result["validation"].to_excel(
        writer,
        sheet_name="validation_vs_benchmark",
        index=False
    )

    result["search_range"].to_excel(
        writer,
        sheet_name="search_range",
        index=False
    )

    result["rf_fixed_features"].to_excel(
        writer,
        sheet_name="rf_fixed_features",
        index=False
    )

    result["lr_fixed_features"].to_excel(
        writer,
        sheet_name="lr_fixed_features",
        index=False
    )

    result["lgbm_fixed_features"].to_excel(
        writer,
        sheet_name="lgbm_fixed_features",
        index=False
    )

    result["similar_batches"].to_excel(
        writer,
        sheet_name="similar_batches",
        index=False
    )

    result["low_gas_batches"].to_excel(
        writer,
        sheet_name="low_gas_batches",
        index=False
    )

    result["rf_top_candidates"].to_excel(
        writer,
        sheet_name="rf_top_candidates",
        index=False
    )

    result["lr_top_candidates"].to_excel(
        writer,
        sheet_name="lr_top_candidates",
        index=False
    )

    result["lgbm_top_candidates"].to_excel(
        writer,
        sheet_name="lgbm_top_candidates",
        index=False
    )

print("\n推荐结果已保存到:", recommend_excel_path)

模型 checkpoint 文件夹: /Users/tian/Desktop/prediction_project/models
推荐结果输出文件夹: /Users/tian/Desktop/prediction_project/recommendation

正在查找模型关键词: ['Random_Forest']
找到候选 checkpoint 数量: 2

候选模型: /Users/tian/Desktop/prediction_project/models/A_解释型全变量版本__Random_Forest_no_y_outlier.joblib
是否文件名包含 B_少共线性版本: False
是否包含 forbidden cols: True
训练特征:
 - 10#熔炼炉总投料重量(kg)
 - 10#熔炼炉固体料重量比例
 - 熔炼炉B当前批次生产时间_PLC
 - 熔炼炉B当前批次熔炼时间_PLC
 - 熔炼炉B当前批次等待时长_PLC
 - 熔炼炉B当前批次炉门打开次数_PLC
 - 熔炼炉B当前批次炉门打开时长_PLC

候选模型: /Users/tian/Desktop/prediction_project/models/B_少共线性版本__Random_Forest_no_y_outlier.joblib
是否文件名包含 B_少共线性版本: True
是否包含 forbidden cols: False
训练特征:
 - 10#熔炼炉总投料重量(kg)
 - 10#熔炼炉固体料重量比例
 - 熔炼炉B当前批次熔炼时间_PLC
 - 熔炼炉B当前批次等待时长_PLC
 - 熔炼炉B当前批次炉门打开次数_PLC
 - 熔炼炉B当前批次炉门打开时长_PLC

最终选择模型: B 版本 checkpoint
模型路径: /Users/tian/Desktop/prediction_project/models/B_少共线性版本__Random_Forest_no_y_outlier.joblib

正在查找模型关键词: ['Linear_Regression', 'Ridge_Regression']
找到候选 checkpoint 数量: 2

候选模型: /Users/tian/Desktop/prediction_project/models/


================ 方法整体气耗对比 ================


,method,gas_metric_type,gas_median,gas_mean,gas_p10,gas_p90,n_records,improvement_vs_all_similar_median_%,improvement_vs_all_similar_mean_%
0,Historical Similar All Batches,actual_all_similar_batches,3681.060000,3712.109579,2876.212000,4595.730000,95,0.000000,0.000000
1,Historical Benchmark Top Low Gas,actual_low_gas_batches,2875.480000,2773.000526,2415.140000,3121.394000,19,21.884457,25.298527
2,Random Forest Optimization,predicted_top_candidates,2974.224554,2974.156646,2972.606596,2975.307047,50,19.201954,19.879611
3,Linear Regression Optimization,predicted_top_candidates,2888.174293,2884.822823,2873.383935,2893.236528,50,21.539603,22.286162
4,LightGBM Optimization,predicted_top_candidates,2616.239896,2612.868116,2595.388169,2624.588880,50,28.926997,29.612312



================ 给工厂看的推荐结果：只包含 4 个可控变量 ================


,推荐变量,推荐方法,推荐值_中位数,推荐范围_p10,推荐范围_p25,推荐范围_p75,推荐范围_p90,Top候选最小值,Top候选最大值
0,10#熔炼炉固体料重量比例,Historical Benchmark,39.590000,27.034000,34.455000,44.920000,49.418000,24.650000,100.000000
1,熔炼炉B当前批次等待时长_PLC,Historical Benchmark,0.670000,0.388000,0.535000,0.785000,1.040000,0.290000,1.950000
2,熔炼炉B当前批次炉门打开次数_PLC,Historical Benchmark,14.000000,8.800000,11.000000,15.000000,16.000000,6.000000,22.000000
3,熔炼炉B当前批次炉门打开时长_PLC,Historical Benchmark,72.000000,39.000000,52.500000,86.500000,100.400000,31.000000,105.000000
5,10#熔炼炉固体料重量比例,Random Forest,30.631432,29.309949,29.660398,31.915349,33.287170,28.289388,33.789653
6,熔炼炉B当前批次等待时长_PLC,Random Forest,0.631481,0.510614,0.532795,0.704512,0.744590,0.460451,0.760394
7,熔炼炉B当前批次炉门打开次数_PLC,Random Forest,11.000000,10.000000,11.000000,11.000000,11.100000,10.000000,12.000000
8,熔炼炉B当前批次炉门打开时长_PLC,Random Forest,53.407012,40.357706,46.006886,64.621747,69.647700,39.311606,106.398381
11,10#熔炼炉固体料重量比例,Linear Regression,27.562559,27.228726,27.326024,27.864902,28.436101,27.060424,29.045893
12,熔炼炉B当前批次等待时长_PLC,Linear Regression,0.471088,0.398781,0.422633,0.522816,0.600756,0.387095,0.721842


...


================ LightGBM Top Candidates ================


,10#熔炼炉总投料重量(kg),10#熔炼炉固体料重量比例,熔炼炉B当前批次等待时长_PLC,熔炼炉B当前批次炉门打开次数_PLC,熔炼炉B当前批次炉门打开时长_PLC,熔炼炉B当前批次熔炼时间_PLC,predicted_gas
4165,86000,32.200749,0.664226,15,104.919378,7.47,2586.867597
44071,86000,28.690166,0.662438,12,102.127272,7.47,2588.350496
41855,86000,29.449623,0.521619,11,102.402071,7.47,2592.383722
14014,86000,31.329438,0.697198,15,106.121455,7.47,2593.466130
19242,86000,35.225154,0.674359,11,101.921134,7.47,2595.388169
27759,86000,34.247992,0.655387,11,102.625788,7.47,2595.388169
48606,86000,29.187837,0.724983,15,122.929807,7.47,2596.263115
6809,86000,29.700767,0.661009,15,119.623900,7.47,2596.298687
25449,86000,31.499192,0.676780,15,105.030710,7.47,2596.910531
19283,86000,29.612164,0.933122,15,106.280142,7.47,2603.751165



================ 历史相似低气耗批次 ================


,batch_id,10#熔炼炉固体料重量比例,10#熔炼炉总投料重量(kg),10#熔炼炉液体料重量比例,熔炼炉B当前批次总气耗_PLC,熔炼炉B当前批次炉门打开时长_PLC,熔炼炉B当前批次炉门打开次数_PLC,熔炼炉B当前批次熔炼时间_PLC,熔炼炉B当前批次生产时间_PLC,熔炼炉B当前批次等待时长_PLC
232,ER266,35.02,87770.0,64.98,1919.34,79.0,11.0,4.45,5.52,0.34
5,ER019,27.23,87176.0,72.77,2155.30,89.0,13.0,6.78,7.82,1.04
244,ER278,38.42,84352.0,61.58,2480.10,80.0,6.0,6.27,7.15,0.72
131,ER164,100.00,86492.0,78.07,2488.93,59.0,12.0,7.98,8.67,0.52
102,ER117,33.89,86581.0,66.11,2615.38,67.0,14.0,6.59,7.37,0.29
284,ER318,38.34,85198.0,61.66,2636.62,105.0,16.0,8.41,9.40,0.72
165,ER198,53.21,87114.0,46.79,2742.39,102.0,15.0,6.13,7.21,0.59
129,ER162,29.49,84013.0,70.51,2772.48,56.0,12.0,8.31,9.06,0.50
279,ER313,47.62,84193.0,52.38,2785.91,31.0,15.0,6.83,8.14,0.55
161,ER194,46.96,84920.0,61.66,2875.48,84.0,11.0,7.67,10.12,1.95



推荐结果已保存到: /Users/tian/Desktop/prediction_project/recommendation/recommendation_total_weight_86000_with_lightgbm.xlsx


In [3]:
# ============================================================
# 16. LightGBM Random Search vs Genetic Algorithm Comparison
#
# 目的：
#   比较 LightGBM 使用 Random Search 和 GA 的寻优表现区别
#
# 对比对象：
#   1. LightGBM Random Search 50000
#      - 来自你原始 result["lgbm_top_candidates"]
#
#   2. LightGBM Random Search 5000
#      - 为了和 GA 做公平比较
#
#   3. LightGBM GA 5000
#      - population_size=100
#      - n_generations=50
#      - 总预测次数约 100 * 50 = 5000
#
# 输出：
#   - 气耗预测对比
#   - 推荐变量对比
#   - 是否落在 historical benchmark p10~p90
#   - GA 每一代收敛过程
#   - Top candidates
#   - Excel 文件
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 16.1 Safe display
# ============================================================

def safe_display_ga(obj):
    """
    兼容 Jupyter Notebook 和普通 .py 脚本。
    """
    try:
        display(obj)
    except NameError:
        print(obj)


# ============================================================
# 16.2 Check required variables
# ============================================================

required_runtime_names = [
    "result",
    "df_no_outlier",
    "lgbm_model",
    "lgbm_feature_cols",
    "factory_input_total_weight",
    "weight_col",
    "target_col",
    "controllable_cols",
    "recommend_output_dir",
]

missing_runtime_names = []

for name in required_runtime_names:
    if name not in globals():
        missing_runtime_names.append(name)

if len(missing_runtime_names) > 0:
    raise NameError(
        "下面这些变量不存在。请确认这段代码放在原推荐系统代码后面运行：\n"
        + str(missing_runtime_names)
    )

print("\n================ LightGBM Random Search vs GA 对比开始 ================")
print("当前总投料重量:", factory_input_total_weight)
print("LightGBM 特征列:")
for c in lgbm_feature_cols:
    print(" -", c)


# ============================================================
# 16.3 GA optimization function
# ============================================================

def genetic_algorithm_optimization_recommendation(
    trained_model,
    df_reference,
    similar_df,
    low_gas_df,
    total_weight,
    feature_cols,
    weight_col,
    target_col,
    controllable_cols,
    population_size=100,
    n_generations=50,
    elite_ratio=0.2,
    mutation_rate=0.15,
    mutation_scale=0.10,
    top_n=50,
    random_state=42,
    range_low_q=0.10,
    range_high_q=0.90,
    range_source="similar",
    fixed_value_strategy="low_gas_median"
):
    """
    遗传算法 GA 版本的模型约束优化推荐。

    个体 individual:
        一组 4 个可控变量

    固定：
        - 总投料重量 total_weight

    优化：
        - controllable_cols 里的 4 个变量

    自动补值：
        - 模型需要但不推荐给工厂的变量

    目标：
        - 最小化 trained_model 预测的 predicted_gas
    """

    rng = np.random.default_rng(random_state)

    # ------------------------------------------------------------
    # 1. Decide search range source
    # ------------------------------------------------------------

    if range_source == "similar":
        range_base_df = similar_df.copy()
    elif range_source == "all":
        range_base_df = df_reference.copy()
    elif range_source == "low_gas":
        range_base_df = low_gas_df.copy()
    else:
        raise ValueError("range_source 只能是 'similar', 'all', 或 'low_gas'")

    if len(range_base_df) == 0:
        raise ValueError("range_base_df 为空，无法计算搜索范围。")

    # ------------------------------------------------------------
    # 2. Define search ranges
    # ------------------------------------------------------------

    range_rows = []
    ranges = {}

    for col in controllable_cols:

        if col not in range_base_df.columns:
            raise ValueError(f"range_base_df 中缺少可控变量列: {col}")

        low = range_base_df[col].quantile(range_low_q)
        high = range_base_df[col].quantile(range_high_q)

        if pd.isna(low) or pd.isna(high):
            raise ValueError(f"{col} 的搜索范围存在 NaN，请检查该列数据。")

        if low > high:
            raise ValueError(f"{col} 的搜索范围异常：low > high。")

        if low == high:
            print(f"警告: {col} 的 search_low 和 search_high 相同，候选值将固定为 {low}")

        ranges[col] = (low, high)

        range_rows.append({
            "variable": col,
            "range_source": range_source,
            "range_low_q": range_low_q,
            "range_high_q": range_high_q,
            "search_low": low,
            "search_high": high,
        })

    range_df = pd.DataFrame(range_rows)

    # ------------------------------------------------------------
    # 3. Fixed values for model-required but non-recommended variables
    # ------------------------------------------------------------

    fixed_values = {}
    fixed_feature_rows = []

    for col in feature_cols:

        if col == weight_col:
            continue

        if col in controllable_cols:
            continue

        if col not in low_gas_df.columns or col not in similar_df.columns:
            raise ValueError(
                f"模型需要变量 {col}，但 low_gas_df 或 similar_df 中没有该列。"
            )

        if fixed_value_strategy == "low_gas_median":
            fixed_value = low_gas_df[col].median()
            fixed_source = "low_gas_median"

        elif fixed_value_strategy == "similar_median":
            fixed_value = similar_df[col].median()
            fixed_source = "similar_median"

        else:
            raise ValueError(
                "fixed_value_strategy 只能是 'low_gas_median' 或 'similar_median'"
            )

        if pd.isna(fixed_value):
            raise ValueError(
                f"模型需要变量 {col}，但是无法计算固定参考值。"
                f"请检查该列在 low_gas_df / similar_df 中是否全是 NaN。"
            )

        fixed_values[col] = fixed_value

        fixed_feature_rows.append({
            "variable": col,
            "fixed_value": fixed_value,
            "fixed_source": fixed_source,
            "reason": "model_required_but_not_recommended",
        })

    fixed_feature_df = pd.DataFrame(fixed_feature_rows)

    # ------------------------------------------------------------
    # 4. Initialize population
    # ------------------------------------------------------------

    def initialize_population():

        population = pd.DataFrame()

        for col in controllable_cols:

            low, high = ranges[col]

            if "次数" in col:
                low_int = int(np.floor(low))
                high_int = int(np.ceil(high))

                if low_int == high_int:
                    population[col] = np.repeat(low_int, population_size)
                else:
                    population[col] = rng.integers(
                        low=low_int,
                        high=high_int + 1,
                        size=population_size
                    )

            else:
                if low == high:
                    population[col] = np.repeat(low, population_size)
                else:
                    population[col] = rng.uniform(
                        low=low,
                        high=high,
                        size=population_size
                    )

        return population

    # ------------------------------------------------------------
    # 5. Evaluate population
    # ------------------------------------------------------------

    def evaluate_population(population, generation):

        candidates = population.copy()

        # 固定总投料重量
        candidates[weight_col] = total_weight

        # 自动补模型需要但不推荐的变量
        for col, value in fixed_values.items():
            candidates[col] = value

        missing_candidate_cols = [
            c for c in feature_cols
            if c not in candidates.columns
        ]

        if len(missing_candidate_cols) > 0:
            raise ValueError(
                f"候选数据中缺少模型需要的特征列: {missing_candidate_cols}"
            )

        candidate_X = candidates[feature_cols].copy()

        candidates["predicted_gas"] = trained_model.predict(candidate_X)
        candidates["generation"] = generation

        return candidates

    # ------------------------------------------------------------
    # 6. Crossover
    # ------------------------------------------------------------

    def crossover(parent1, parent2):

        child = {}

        for col in controllable_cols:

            # 炉门打开次数：整数变量，从父代中随机继承
            if "次数" in col:
                if rng.random() < 0.5:
                    child[col] = parent1[col]
                else:
                    child[col] = parent2[col]

            # 其他连续变量：父母加权平均
            else:
                alpha = rng.uniform(0, 1)
                child[col] = alpha * parent1[col] + (1 - alpha) * parent2[col]

        return child

    # ------------------------------------------------------------
    # 7. Mutation
    # ------------------------------------------------------------

    def mutate(child):

        for col in controllable_cols:

            if rng.random() >= mutation_rate:
                continue

            low, high = ranges[col]

            # 整数变量 mutation
            if "次数" in col:

                low_int = int(np.floor(low))
                high_int = int(np.ceil(high))

                if low_int < high_int:
                    child[col] = rng.integers(
                        low=low_int,
                        high=high_int + 1
                    )

            # 连续变量 mutation
            else:

                width = high - low

                if width > 0:
                    noise = rng.normal(
                        loc=0,
                        scale=mutation_scale * width
                    )

                    child[col] = child[col] + noise
                    child[col] = np.clip(child[col], low, high)

        return child

    # ------------------------------------------------------------
    # 8. GA main loop
    # ------------------------------------------------------------

    population = initialize_population()

    all_evaluated_candidates = []
    generation_log_rows = []

    elite_count = max(2, int(population_size * elite_ratio))

    for generation in range(n_generations):

        evaluated = evaluate_population(
            population=population,
            generation=generation
        )

        evaluated = evaluated.sort_values(
            "predicted_gas",
            ascending=True
        ).reset_index(drop=True)

        all_evaluated_candidates.append(evaluated.copy())

        generation_log_rows.append({
            "generation": generation,
            "best_predicted_gas": evaluated["predicted_gas"].min(),
            "median_predicted_gas": evaluated["predicted_gas"].median(),
            "mean_predicted_gas": evaluated["predicted_gas"].mean(),
            "worst_predicted_gas": evaluated["predicted_gas"].max(),
        })

        elites = evaluated.head(elite_count).copy()

        parent_pool = elites[controllable_cols].reset_index(drop=True)

        next_population_rows = []

        # 精英保留
        for _, row in parent_pool.iterrows():
            next_population_rows.append(row.to_dict())

        # 交叉 + 变异生成下一代剩余个体
        while len(next_population_rows) < population_size:

            parent_indices = rng.choice(
                len(parent_pool),
                size=2,
                replace=True
            )

            parent1 = parent_pool.iloc[parent_indices[0]]
            parent2 = parent_pool.iloc[parent_indices[1]]

            child = crossover(parent1, parent2)
            child = mutate(child)

            next_population_rows.append(child)

        population = pd.DataFrame(next_population_rows)

        # 保证所有变量仍然在范围内
        for col in controllable_cols:

            low, high = ranges[col]

            if "次数" in col:
                low_int = int(np.floor(low))
                high_int = int(np.ceil(high))

                population[col] = (
                    population[col]
                    .round()
                    .clip(low_int, high_int)
                    .astype(int)
                )

            else:
                population[col] = population[col].clip(low, high)

    # ------------------------------------------------------------
    # 9. Collect all candidates
    # ------------------------------------------------------------

    all_candidates = pd.concat(
        all_evaluated_candidates,
        ignore_index=True
    )

    generation_log_df = pd.DataFrame(generation_log_rows)

    top_candidates = all_candidates.sort_values(
        "predicted_gas",
        ascending=True
    ).head(top_n).copy()

    # ------------------------------------------------------------
    # 10. Recommendation summary
    # ------------------------------------------------------------

    summary_rows = []

    fixed_cols = [
        c for c in feature_cols
        if c not in controllable_cols and c != weight_col
    ]

    summary_cols = controllable_cols + fixed_cols + ["predicted_gas"]

    for col in summary_cols:

        summary_rows.append({
            "variable": col,
            "method": "genetic_algorithm_optimization",
            "value_median": top_candidates[col].median(),
            "value_mean": top_candidates[col].mean(),
            "value_p10": top_candidates[col].quantile(0.10),
            "value_p25": top_candidates[col].quantile(0.25),
            "value_p75": top_candidates[col].quantile(0.75),
            "value_p90": top_candidates[col].quantile(0.90),
            "value_min": top_candidates[col].min(),
            "value_max": top_candidates[col].max(),
            "population_size": population_size,
            "n_generations": n_generations,
            "elite_ratio": elite_ratio,
            "mutation_rate": mutation_rate,
            "mutation_scale": mutation_scale,
            "top_n": top_n,
        })

    recommendation_summary = pd.DataFrame(summary_rows)

    return (
        all_candidates,
        top_candidates,
        recommendation_summary,
        range_df,
        fixed_feature_df,
        generation_log_df
    )


# ============================================================
# 16.4 Validation function for GA comparison
# ============================================================

def validate_recommendation_against_benchmark_ga(
    model_summary,
    benchmark_summary,
    controllable_cols
):
    """
    检查模型推荐的 4 个可控变量中位数，
    是否落在历史相似低气耗批次 p10~p90 区间内。
    """

    rows = []

    for col in controllable_cols:

        model_row = model_summary[
            model_summary["variable"] == col
        ].iloc[0]

        bench_row = benchmark_summary[
            benchmark_summary["variable"] == col
        ].iloc[0]

        model_median = model_row["value_median"]
        bench_p10 = bench_row["value_p10"]
        bench_p90 = bench_row["value_p90"]
        bench_median = bench_row["value_median"]

        in_benchmark_range = (
            model_median >= bench_p10
            and model_median <= bench_p90
        )

        rows.append({
            "variable": col,
            "model_recommended_median": model_median,
            "benchmark_median": bench_median,
            "benchmark_p10": bench_p10,
            "benchmark_p90": bench_p90,
            "model_median_in_benchmark_p10_p90": in_benchmark_range,
            "difference_vs_benchmark_median": model_median - bench_median,
        })

    return pd.DataFrame(rows)


# ============================================================
# 16.5 Helper: score top candidates
# ============================================================

def summarize_top_candidate_score(top_df, method_name):

    return {
        "method": method_name,
        "gas_metric_type": "predicted_top_candidates",
        "gas_median": top_df["predicted_gas"].median(),
        "gas_mean": top_df["predicted_gas"].mean(),
        "gas_p10": top_df["predicted_gas"].quantile(0.10),
        "gas_p25": top_df["predicted_gas"].quantile(0.25),
        "gas_p75": top_df["predicted_gas"].quantile(0.75),
        "gas_p90": top_df["predicted_gas"].quantile(0.90),
        "gas_min": top_df["predicted_gas"].min(),
        "gas_max": top_df["predicted_gas"].max(),
        "n_records": len(top_df),
    }


# ============================================================
# 16.6 Run fair LightGBM Random Search 5000
# ============================================================

print("\n================ Running LightGBM Random Search 5000 ================")

(
    lgbm_random_5000_candidates,
    lgbm_random_5000_top,
    lgbm_random_5000_summary,
    lgbm_random_5000_search_range,
    lgbm_random_5000_fixed_features
) = model_based_optimization_recommendation(
    trained_model=lgbm_model,
    df_reference=df_no_outlier,
    similar_df=result["similar_batches"],
    low_gas_df=result["low_gas_batches"],
    total_weight=factory_input_total_weight,
    feature_cols=lgbm_feature_cols,
    weight_col=weight_col,
    target_col=target_col,
    controllable_cols=controllable_cols,

    n_candidates=5000,
    top_n=50,
    random_state=42,
    range_source="similar",
    fixed_value_strategy="low_gas_median"
)

lgbm_random_5000_summary["model"] = "LightGBM Random Search 5000"


# ============================================================
# 16.7 Run LightGBM GA
# ============================================================

print("\n================ Running LightGBM GA ================")

(
    lgbm_ga_candidates,
    lgbm_ga_top,
    lgbm_ga_summary,
    lgbm_ga_search_range,
    lgbm_ga_fixed_features,
    lgbm_ga_generation_log
) = genetic_algorithm_optimization_recommendation(
    trained_model=lgbm_model,
    df_reference=df_no_outlier,
    similar_df=result["similar_batches"],
    low_gas_df=result["low_gas_batches"],
    total_weight=factory_input_total_weight,
    feature_cols=lgbm_feature_cols,
    weight_col=weight_col,
    target_col=target_col,
    controllable_cols=controllable_cols,

    population_size=100,
    n_generations=50,
    elite_ratio=0.2,
    mutation_rate=0.15,
    mutation_scale=0.10,

    top_n=50,
    random_state=42,
    range_source="similar",
    fixed_value_strategy="low_gas_median"
)

lgbm_ga_summary["model"] = "LightGBM GA"


# ============================================================
# 16.8 Prepare original LightGBM Random Search 50000 result
# ============================================================

lgbm_random_50000_top = result["lgbm_top_candidates"].copy()

lgbm_random_50000_summary = result["combined_summary"][
    result["combined_summary"]["model"] == "LightGBM"
].copy()

lgbm_random_50000_summary["model"] = "LightGBM Random Search 50000"


# ============================================================
# 16.9 Score comparison
# ============================================================

lgbm_random_vs_ga_score = pd.DataFrame([
    summarize_top_candidate_score(
        lgbm_random_50000_top,
        "LightGBM Random Search 50000"
    ),
    summarize_top_candidate_score(
        lgbm_random_5000_top,
        "LightGBM Random Search 5000"
    ),
    summarize_top_candidate_score(
        lgbm_ga_top,
        "LightGBM GA 5000"
    ),
])

baseline_random_5000_median = lgbm_random_vs_ga_score.loc[
    lgbm_random_vs_ga_score["method"] == "LightGBM Random Search 5000",
    "gas_median"
].iloc[0]

baseline_random_5000_mean = lgbm_random_vs_ga_score.loc[
    lgbm_random_vs_ga_score["method"] == "LightGBM Random Search 5000",
    "gas_mean"
].iloc[0]

baseline_random_50000_median = lgbm_random_vs_ga_score.loc[
    lgbm_random_vs_ga_score["method"] == "LightGBM Random Search 50000",
    "gas_median"
].iloc[0]

baseline_random_50000_mean = lgbm_random_vs_ga_score.loc[
    lgbm_random_vs_ga_score["method"] == "LightGBM Random Search 50000",
    "gas_mean"
].iloc[0]

lgbm_random_vs_ga_score["improvement_vs_random_5000_median_%"] = (
    baseline_random_5000_median - lgbm_random_vs_ga_score["gas_median"]
) / baseline_random_5000_median * 100

lgbm_random_vs_ga_score["improvement_vs_random_5000_mean_%"] = (
    baseline_random_5000_mean - lgbm_random_vs_ga_score["gas_mean"]
) / baseline_random_5000_mean * 100

lgbm_random_vs_ga_score["improvement_vs_random_50000_median_%"] = (
    baseline_random_50000_median - lgbm_random_vs_ga_score["gas_median"]
) / baseline_random_50000_median * 100

lgbm_random_vs_ga_score["improvement_vs_random_50000_mean_%"] = (
    baseline_random_50000_mean - lgbm_random_vs_ga_score["gas_mean"]
) / baseline_random_50000_mean * 100


print("\n================ LightGBM Random Search vs GA: 气耗预测对比 ================")
safe_display_ga(lgbm_random_vs_ga_score)


# ============================================================
# 16.10 Variable recommendation comparison
# ============================================================

lgbm_random_ga_variable_compare = pd.concat(
    [
        lgbm_random_50000_summary,
        lgbm_random_5000_summary,
        lgbm_ga_summary,
    ],
    ignore_index=True
)

lgbm_random_ga_variable_compare = lgbm_random_ga_variable_compare[
    lgbm_random_ga_variable_compare["variable"].isin(
        controllable_cols + ["predicted_gas"]
    )
][
    [
        "variable",
        "model",
        "value_median",
        "value_mean",
        "value_p10",
        "value_p25",
        "value_p75",
        "value_p90",
        "value_min",
        "value_max",
    ]
].sort_values(["variable", "model"])

print("\n================ LightGBM Random Search vs GA: 推荐变量对比 ================")
safe_display_ga(lgbm_random_ga_variable_compare)


# ============================================================
# 16.11 Benchmark validation comparison
# ============================================================

lgbm_random_50000_validation = result["validation"][
    result["validation"]["model"] == "LightGBM"
].copy()

lgbm_random_50000_validation["model"] = "LightGBM Random Search 50000"

lgbm_random_5000_validation = validate_recommendation_against_benchmark_ga(
    model_summary=lgbm_random_5000_summary,
    benchmark_summary=result["benchmark_summary"],
    controllable_cols=controllable_cols
)

lgbm_random_5000_validation["model"] = "LightGBM Random Search 5000"

lgbm_ga_validation = validate_recommendation_against_benchmark_ga(
    model_summary=lgbm_ga_summary,
    benchmark_summary=result["benchmark_summary"],
    controllable_cols=controllable_cols
)

lgbm_ga_validation["model"] = "LightGBM GA 5000"

lgbm_random_vs_ga_validation = pd.concat(
    [
        lgbm_random_50000_validation,
        lgbm_random_5000_validation,
        lgbm_ga_validation,
    ],
    ignore_index=True
)

print("\n================ LightGBM Random Search vs GA: 是否落在历史 benchmark 范围内 ================")
safe_display_ga(lgbm_random_vs_ga_validation)


# ============================================================
# 16.12 GA generation log
# ============================================================

print("\n================ LightGBM GA 每一代优化过程 ================")
safe_display_ga(lgbm_ga_generation_log)


# ============================================================
# 16.13 Top candidates preview
# ============================================================

print("\n================ LightGBM Random Search 50000 Top Candidates ================")
safe_display_ga(lgbm_random_50000_top.head(10))

print("\n================ LightGBM Random Search 5000 Top Candidates ================")
safe_display_ga(lgbm_random_5000_top.head(10))

print("\n================ LightGBM GA Top Candidates ================")
safe_display_ga(lgbm_ga_top.head(10))


# ============================================================
# 16.14 Simple conclusion table
# ============================================================

best_by_median = lgbm_random_vs_ga_score.sort_values(
    "gas_median",
    ascending=True
).head(1)["method"].iloc[0]

best_by_mean = lgbm_random_vs_ga_score.sort_values(
    "gas_mean",
    ascending=True
).head(1)["method"].iloc[0]

conclusion_rows = [
    {
        "metric": "lowest_gas_median",
        "best_method": best_by_median,
    },
    {
        "metric": "lowest_gas_mean",
        "best_method": best_by_mean,
    },
]

lgbm_random_vs_ga_conclusion = pd.DataFrame(conclusion_rows)

print("\n================ LightGBM Random Search vs GA: 简单结论 ================")
safe_display_ga(lgbm_random_vs_ga_conclusion)


# ============================================================
# 16.15 Export comparison result to Excel
# ============================================================

lgbm_ga_compare_excel_path = os.path.join(
    recommend_output_dir,
    f"lightgbm_random_vs_ga_total_weight_{int(factory_input_total_weight)}.xlsx"
)

with pd.ExcelWriter(lgbm_ga_compare_excel_path, engine="openpyxl") as writer:

    lgbm_random_vs_ga_score.to_excel(
        writer,
        sheet_name="score_compare",
        index=False
    )

    lgbm_random_ga_variable_compare.to_excel(
        writer,
        sheet_name="variable_compare",
        index=False
    )

    lgbm_random_vs_ga_validation.to_excel(
        writer,
        sheet_name="validation",
        index=False
    )

    lgbm_ga_generation_log.to_excel(
        writer,
        sheet_name="ga_generation_log",
        index=False
    )

    lgbm_random_vs_ga_conclusion.to_excel(
        writer,
        sheet_name="simple_conclusion",
        index=False
    )

    lgbm_ga_search_range.to_excel(
        writer,
        sheet_name="ga_search_range",
        index=False
    )

    lgbm_ga_fixed_features.to_excel(
        writer,
        sheet_name="ga_fixed_features",
        index=False
    )

    lgbm_random_5000_search_range.to_excel(
        writer,
        sheet_name="random_5000_search_range",
        index=False
    )

    lgbm_random_5000_fixed_features.to_excel(
        writer,
        sheet_name="random_5000_fixed_features",
        index=False
    )

    lgbm_random_50000_top.to_excel(
        writer,
        sheet_name="random_50000_top",
        index=False
    )

    lgbm_random_5000_top.to_excel(
        writer,
        sheet_name="random_5000_top",
        index=False
    )

    lgbm_ga_top.to_excel(
        writer,
        sheet_name="ga_top",
        index=False
    )

    lgbm_ga_candidates.to_excel(
        writer,
        sheet_name="ga_all_candidates",
        index=False
    )

print("\nLightGBM Random Search vs GA 对比结果已保存到:")
print(lgbm_ga_compare_excel_path)

print("\n================ LightGBM Random Search vs GA 对比结束 ================")


================ LightGBM Random Search vs GA 对比开始 ================
当前总投料重量: 86000
LightGBM 特征列:
 - 10#熔炼炉总投料重量(kg)
 - 10#熔炼炉固体料重量比例
 - 熔炼炉B当前批次熔炼时间_PLC
 - 熔炼炉B当前批次等待时长_PLC
 - 熔炼炉B当前批次炉门打开次数_PLC
 - 熔炼炉B当前批次炉门打开时长_PLC

================ Running LightGBM Random Search 5000 ================

================ Running LightGBM GA ================



================ LightGBM Random Search vs GA: 气耗预测对比 ================


,method,gas_metric_type,gas_median,gas_mean,gas_p10,gas_p25,gas_p75,gas_p90,gas_min,gas_max,n_records,improvement_vs_random_5000_median_%,improvement_vs_random_5000_mean_%,improvement_vs_random_50000_median_%,improvement_vs_random_50000_mean_%
0,LightGBM Random Search 50000,predicted_top_candidates,2616.239896,2612.868116,2595.388169,2607.346811,2620.436991,2624.588880,2586.867597,2625.598827,50,2.215203,2.135097,0.000000,0.000000
1,LightGBM Random Search 5000,predicted_top_candidates,2675.507830,2669.872488,2633.838240,2651.578828,2693.281214,2699.219492,2602.335290,2705.189652,50,0.000000,0.000000,-2.265386,-2.181678
2,LightGBM GA 5000,predicted_top_candidates,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,50,3.402788,3.198899,1.214489,1.087011



================ LightGBM Random Search vs GA: 推荐变量对比 ================


,variable,model,value_median,value_mean,value_p10,value_p25,value_p75,value_p90,value_min,value_max
12,10#熔炼炉固体料重量比例,LightGBM GA,34.409815,34.472260,34.284204,34.322925,34.533395,34.588727,34.237797,35.640894
6,10#熔炼炉固体料重量比例,LightGBM Random Search 5000,31.152084,31.403161,28.585463,29.367383,33.612311,34.942959,27.078256,35.433007
0,10#熔炼炉固体料重量比例,LightGBM Random Search 50000,31.727105,31.912246,28.935996,29.820823,34.032092,34.900074,27.119065,35.761560
17,predicted_gas,LightGBM GA,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958,2584.465958
11,predicted_gas,LightGBM Random Search 5000,2675.507830,2669.872488,2633.838240,2651.578828,2693.281214,2699.219492,2602.335290,2705.189652
5,predicted_gas,LightGBM Random Search 50000,2616.239896,2612.868116,2595.388169,2607.346811,2620.436991,2624.588880,2586.867597,2625.598827
15,熔炼炉B当前批次炉门打开时长_PLC,LightGBM GA,105.455679,105.739339,104.695032,104.991546,106.057962,106.675244,104.448780,108.969071
9,熔炼炉B当前批次炉门打开时长_PLC,LightGBM Random Search 5000,101.638876,95.267537,56.895409,78.316008,114.419552,121.857165,41.969518,128.403216
3,熔炼炉B当前批次炉门打开时长_PLC,LightGBM Random Search 50000,105.933612,109.266611,102.139221,102.961365,112.539498,123.237912,100.555568,129.004414
14,熔炼炉B当前批次炉门打开次数_PLC,LightGBM GA,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000


...


================ LightGBM GA Top Candidates ================


,10#熔炼炉固体料重量比例,熔炼炉B当前批次等待时长_PLC,熔炼炉B当前批次炉门打开次数_PLC,熔炼炉B当前批次炉门打开时长_PLC,10#熔炼炉总投料重量(kg),熔炼炉B当前批次熔炼时间_PLC,predicted_gas,generation
2325,34.336841,0.660855,15,105.828814,86000,7.47,2584.465958,23
2847,34.658017,0.660571,15,104.448780,86000,7.47,2584.465958,28
2848,34.563430,0.659376,15,106.832310,86000,7.47,2584.465958,28
2849,34.321097,0.661599,15,106.563296,86000,7.47,2584.465958,28
2850,34.426860,0.660937,15,104.833199,86000,7.47,2584.465958,28
2851,34.389474,0.660791,15,105.733806,86000,7.47,2584.465958,28
2852,34.283800,0.660585,15,104.568234,86000,7.47,2584.465958,28
2853,34.237797,0.660971,15,106.535540,86000,7.47,2584.465958,28
2854,34.420950,0.660080,15,104.790138,86000,7.47,2584.465958,28
2855,34.361377,0.660979,15,105.380062,86000,7.47,2584.465958,28



================ LightGBM Random Search vs GA: 简单结论 ================


,metric,best_method
0,lowest_gas_median,LightGBM GA 5000
1,lowest_gas_mean,LightGBM GA 5000



LightGBM Random Search vs GA 对比结果已保存到:
/Users/tian/Desktop/prediction_project/recommendation/lightgbm_random_vs_ga_total_weight_86000.xlsx

================ LightGBM Random Search vs GA 对比结束 ================


In [4]:
print("GA Top 50 shape:", lgbm_ga_top.shape)

print("GA Top 50 去重后数量:")
print(
    lgbm_ga_top[
        controllable_cols + ["predicted_gas"]
    ].drop_duplicates().shape[0]
)

safe_display(
    lgbm_ga_top[
        controllable_cols + ["predicted_gas", "generation"]
    ].drop_duplicates().head(20)
)
print("\n================ GA 搜索范围 ================")
safe_display_ga(lgbm_ga_search_range)
print("\n================ LightGBM GA Top 50 变量分布 ================")

ga_top_variable_distribution = lgbm_ga_top[
    controllable_cols + ["predicted_gas", "generation"]
].describe()

safe_display_ga(ga_top_variable_distribution)
def check_ga_top_boundary_status(
    ga_top,
    search_range_df,
    controllable_cols
):
    rows = []

    for col in controllable_cols:

        range_row = search_range_df[
            search_range_df["variable"] == col
        ].iloc[0]

        search_low = range_row["search_low"]
        search_high = range_row["search_high"]

        top_min = ga_top[col].min()
        top_max = ga_top[col].max()
        top_median = ga_top[col].median()

        width = search_high - search_low

        if width == 0:
            median_position_ratio = np.nan
            near_low_boundary = True
            near_high_boundary = True
        else:
            median_position_ratio = (top_median - search_low) / width
            near_low_boundary = median_position_ratio <= 0.10
            near_high_boundary = median_position_ratio >= 0.90

        rows.append({
            "variable": col,
            "search_low": search_low,
            "search_high": search_high,
            "top_min": top_min,
            "top_max": top_max,
            "top_median": top_median,
            "median_position_ratio": median_position_ratio,
            "near_low_boundary": near_low_boundary,
            "near_high_boundary": near_high_boundary,
        })

    return pd.DataFrame(rows)


ga_boundary_check = check_ga_top_boundary_status(
    ga_top=lgbm_ga_top,
    search_range_df=lgbm_ga_search_range,
    controllable_cols=controllable_cols
)

print("\n================ LightGBM GA Top 50 是否贴边 ================")
safe_display_ga(ga_boundary_check)


GA Top 50 shape: (50, 8)
GA Top 50 去重后数量:
47


,10#熔炼炉固体料重量比例,熔炼炉B当前批次等待时长_PLC,熔炼炉B当前批次炉门打开次数_PLC,熔炼炉B当前批次炉门打开时长_PLC,predicted_gas,generation
2325,34.336841,0.660855,15,105.828814,2584.465958,23
2847,34.658017,0.660571,15,104.448780,2584.465958,28
2848,34.563430,0.659376,15,106.832310,2584.465958,28
2849,34.321097,0.661599,15,106.563296,2584.465958,28
2850,34.426860,0.660937,15,104.833199,2584.465958,28
2851,34.389474,0.660791,15,105.733806,2584.465958,28
2852,34.283800,0.660585,15,104.568234,2584.465958,28
2853,34.237797,0.660971,15,106.535540,2584.465958,28
2854,34.420950,0.660080,15,104.790138,2584.465958,28
2855,34.361377,0.660979,15,105.380062,2584.465958,28



================ GA 搜索范围 ================


,variable,range_source,range_low_q,range_high_q,search_low,search_high
0,10#熔炼炉固体料重量比例,similar,0.1,0.9,32.702,54.034
1,熔炼炉B当前批次等待时长_PLC,similar,0.1,0.9,0.428,1.772
2,熔炼炉B当前批次炉门打开次数_PLC,similar,0.1,0.9,9.000,18.000
3,熔炼炉B当前批次炉门打开时长_PLC,similar,0.1,0.9,45.600,130.200



================ LightGBM GA Top 50 变量分布 ================


,10#熔炼炉固体料重量比例,熔炼炉B当前批次等待时长_PLC,熔炼炉B当前批次炉门打开次数_PLC,熔炼炉B当前批次炉门打开时长_PLC,predicted_gas,generation
count,50.000000,50.000000,50.0,50.000000,5.000000e+01,50.000000
mean,34.472260,0.660504,15.0,105.739339,2.584466e+03,29.040000
std,0.270962,0.000696,0.0,1.054972,1.837457e-12,4.637733
min,34.237797,0.659155,15.0,104.448780,2.584466e+03,23.000000
25%,34.322925,0.660136,15.0,104.991546,2.584466e+03,28.000000
50%,34.409815,0.660626,15.0,105.455679,2.584466e+03,28.000000
75%,34.533395,0.660962,15.0,106.057962,2.584466e+03,28.000000
max,35.640894,0.662108,15.0,108.969071,2.584466e+03,47.000000



================ LightGBM GA Top 50 是否贴边 ================


,variable,search_low,search_high,top_min,top_max,top_median,median_position_ratio,near_low_boundary,near_high_boundary
0,10#熔炼炉固体料重量比例,32.702,54.034,34.237797,35.640894,34.409815,0.080059,True,False
1,熔炼炉B当前批次等待时长_PLC,0.428,1.772,0.659155,0.662108,0.660626,0.173085,False,False
2,熔炼炉B当前批次炉门打开次数_PLC,9.000,18.000,15.000000,15.000000,15.000000,0.666667,False,False
3,熔炼炉B当前批次炉门打开时长_PLC,45.600,130.200,104.448780,108.969071,105.455679,0.707514,False,False
